In [3]:

import joblib
import pandas as pd

from canids.io import load_sample
from canids.features import frame_features

root = "../data/raw/can-train-and-test/set_01"

# Reproduce the exact Test 01 sample
df = load_sample(
    root,
    "test_01_known_vehicle_known_attack",
    fraction=0.005,
    seed=2026
)

# Load the previously trained Random Forest
rf = joblib.load("../models/random_forest.joblib")

# Predict using the same feature extraction pipeline
df["prediction"] = rf.predict(frame_features(df))

# Identify different error types
df["detected"] = (
    (df["attack"] == 1) &
    (df["prediction"] == 1)
).astype(int)

df["missed"] = (
    (df["attack"] == 1) &
    (df["prediction"] == 0)
).astype(int)

df["false_alarm"] = (
    (df["attack"] == 0) &
    (df["prediction"] == 1)
).astype(int)

# Summarize performance for each capture
report = df.groupby("source_file").agg(
    total_frames=("attack", "size"),
    attack_frames=("attack", "sum"),
    detected_attacks=("detected", "sum"),
    missed_attacks=("missed", "sum"),
    false_alarms=("false_alarm", "sum")
)

report["attack_recall"] = (
    report["detected_attacks"] /
    report["attack_frames"].replace(0, float("nan"))
)

display(report.sort_values(
    "missed_attacks", ascending=False
))


  reading test_01_known_vehicle_known_attack\DoS-3.csv
  reading test_01_known_vehicle_known_attack\DoS-4.csv
  reading test_01_known_vehicle_known_attack\force-neutral-3.csv
  reading test_01_known_vehicle_known_attack\force-neutral-4.csv
  reading test_01_known_vehicle_known_attack\rpm-3.csv
  reading test_01_known_vehicle_known_attack\rpm-4.csv
  reading test_01_known_vehicle_known_attack\standstill-3.csv
  reading test_01_known_vehicle_known_attack\standstill-4.csv


,total_frames,attack_frames,detected_attacks,missed_attacks,false_alarms,attack_recall
source_file,,,,,,
test_01_known_vehicle_known_attack/DoS-4.csv,4531,200,0,200,0,0.0
test_01_known_vehicle_known_attack/rpm-3.csv,4010,13,0,13,0,0.0
test_01_known_vehicle_known_attack/rpm-4.csv,4089,1,0,1,0,0.0
test_01_known_vehicle_known_attack/DoS-3.csv,754,92,92,0,0,1.0
test_01_known_vehicle_known_attack/force-neutral-4.csv,4182,3,3,0,0,1.0
test_01_known_vehicle_known_attack/force-neutral-3.csv,4472,5,5,0,0,1.0
test_01_known_vehicle_known_attack/standstill-3.csv,3149,6,6,0,0,1.0
test_01_known_vehicle_known_attack/standstill-4.csv,3016,1,1,0,0,1.0


In [4]:

# Compare CAN identifiers used in DoS-3 and DoS-4

dos = df[
    df["source_file"].str.contains(
        r"DoS-[34]\.csv$", regex=True
    )
].copy()

# We only want injected attack frames
dos_attacks = dos[dos["attack"] == 1].copy()

dos_attacks["capture"] = (
    dos_attacks["source_file"]
    .str.split("/")
    .str[-1]
)

# Examine CAN IDs associated with injected frames
id_comparison = (
    dos_attacks
    .groupby(["capture", "arbitration_id"])
    .agg(
        attack_frames=("attack", "size"),
        detected=("prediction", "sum")
    )
    .reset_index()
)

display(id_comparison)

# Compare common attack payload values
for capture, group in dos_attacks.groupby("capture"):
    print(f"\n{capture}:")
    print("Most frequent attack payloads:")
    print(group["data_field"].value_counts().head(10))


,capture,arbitration_id,attack_frames,detected
0,DoS-3.csv,0C1,92,92
1,DoS-4.csv,0C1,200,0



DoS-3.csv:
Most frequent attack payloads:
data_field
0000000000000000    92
Name: count, dtype: Int64

DoS-4.csv:
Most frequent attack payloads:
data_field
2026662920DA1D72    200
Name: count, dtype: Int64


In [5]:

# Compare attack payloads in DoS-3 and DoS-4

attacks = df[
    (df["attack"] == 1) &
    (df["source_file"].str.endswith(
        ("DoS-3.csv", "DoS-4.csv")
    ))
].copy()

attacks["capture"] = (
    attacks["source_file"].str.split("/").str[-1]
)

attacks["dlc"] = (
    attacks["data_field"].str.len() // 2
)

# Compare payload lengths
display(
    pd.crosstab(attacks["capture"], attacks["dlc"])
)

# Compare payload values
for capture, group in attacks.groupby("capture"):
    print(f"\n{capture}")
    print("Unique payloads:", group["data_field"].nunique())
    display(group["data_field"].value_counts().head(10))


dlc,8
capture,
DoS-3.csv,92
DoS-4.csv,200



DoS-3.csv
Unique payloads: 1


data_field
0000000000000000    92
Name: count, dtype: Int64


DoS-4.csv
Unique payloads: 1


data_field
2026662920DA1D72    200
Name: count, dtype: Int64

In [6]:
for capture, group in attacks.groupby("capture"):
    print(f"\n=== {capture} ===")

    print("Unique payloads:", group["data_field"].nunique())

    print("\nMost common payloads:")
    print(group["data_field"].value_counts().head(10))


=== DoS-3.csv ===
Unique payloads: 1

Most common payloads:
data_field
0000000000000000    92
Name: count, dtype: Int64

=== DoS-4.csv ===
Unique payloads: 1

Most common payloads:
data_field
2026662920DA1D72    200
Name: count, dtype: Int64


In [ ]:

from canids.io import load_sample

root = "../data/raw/can-train-and-test/set_01"

# Load the same training sample used by our experiment
train_df = load_sample(
    root,
    "train_01",
    fraction=0.25,
    seed=42
)

# Filter injected frames with CAN ID 0x0C1
train_dos = train_df[
    (train_df["attack"] == 1) &
    (
        train_df["arbitration_id"]
        .str.upper().str.replace("0X", "", regex=False)
        == "0C1"
    )
].copy()

# Compare payload values in the training data
print("Training attack payloads for CAN ID 0x0C1:")

display(
    train_dos.groupby(
        ["source_file", "data_field"]
    ).size().reset_index(name="frames")
)

# Check whether our two test payloads occurred
for payload in [
    "0000000000000000",
    "2026662920DA1D72"
]:
    count = (
        train_dos["data_field"].str.upper() == payload
    ).sum()

    print(f"{payload}: {count} sampled training attacks")


  reading train_01\accessory-1.csv
  reading train_01\accessory-2.csv
  reading train_01\attack-free-1.csv
  reading train_01\attack-free-2.csv
  reading train_01\DoS-1.csv
  reading train_01\DoS-2.csv
  reading train_01\force-neutral-1.csv
  reading train_01\force-neutral-2.csv
  reading train_01\rpm-1.csv
  reading train_01\rpm-2.csv
  reading train_01\standstill-1.csv
  reading train_01\standstill-2.csv
Training attack payloads for CAN ID 0x0C1:


,source_file,data_field,frames
0,train_01/DoS-2.csv,0000000000000000,528


0000000000000000: 528 sampled training attacks
2026662920DA1D72: 0 sampled training attacks


In [10]:

import pandas as pd
import joblib
from canids.features import frame_features

rf = joblib.load("../models/random_forest.joblib")

# Keep CAN ID and DLC identical.
# Change only the payload.
comparison = pd.DataFrame({
    "arbitration_id": ["0C1", "0C1"],
    "data_field": [
        "0000000000000000",
        "2026662920DA1D72"
    ]
})

features = frame_features(comparison)

comparison["prediction"] = rf.predict(features)

comparison["attack_probability"] = (
    rf.predict_proba(features)[:, 1]
)

display(comparison)


,arbitration_id,data_field,prediction,attack_probability
0,0C1,0000000000000000,1,0.99828
1,0C1,2026662920DA1D72,0,0.00000


In [11]:

# Compare both payloads across normal and attack training frames

payloads = [
    "0000000000000000",
    "2026662920DA1D72"
]

frames = train_df[
    (train_df["arbitration_id"].str.upper() == "0C1") &
    (train_df["data_field"].str.upper().isin(payloads))
].copy()

summary = pd.crosstab(
    frames["data_field"].str.upper(),
    frames["attack"]
)

summary = summary.rename(
    columns={0: "Normal frames", 1: "Attack frames"}
).reindex(payloads, fill_value=0)

display(summary)


attack,Normal frames,Attack frames
data_field,,
0000000000000000,99,528
2026662920DA1D72,0,0
